<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/FindDuplicatesOpenSource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import hashlib

def generate_hash_key(s1, s2, s3):
    # Join with a delimiter to ensure uniqueness
    combined_string = f"{s1}|{s2}|{s3}"

    # Create the SHA-256 hash and return as hex
    return hashlib.sha256(combined_string.encode('utf-8')).hexdigest()



In [17]:
# Usage
strValOne="Sai"
strValTwo="Charan"
strValThree="Anche"
strOriginalRecords=[]
strDuplicateRecords=[]



In [18]:
#Generate a unique identifier for every record. Even a minor variation in a value will generate a new ID
key = generate_hash_key(strValOne, strValTwo, strValThree)
strRecord=[strValOne,strValTwo,strValThree,key]



In [ ]:
#Now Generate the Embeddings to convert the strings into vector. Minor variation will result in a "similar" vector
from sentence_transformers import SentenceTransformer

# 1. Load a pre-trained model (e.g., 'all-MiniLM-L6-v2' is fast and accurate)
model = SentenceTransformer('all-MiniLM-L6-v2')





In [ ]:
!pip install lancedb

In [21]:
import lancedb
from lancedb.pydantic import LanceModel, Vector

# Define a schema with a vector and your custom hashkey
class MyRecord(LanceModel):
    hashkey: str  # The hash you generated earlier
    vector: Vector(384)  # Size must match your embedding model (e.g., 384 for MiniLM)
    #text: str # Optional: keep the original text

# Connect to the database
db = lancedb.connect("./lancedb_data")

# Create the table with the schema
table = db.create_table("my_table", schema=MyRecord, mode="overwrite")

In [ ]:
# 2. Define your 3 strings in a list
strings = [strValOne+strValTwo+strValThree]

# 3. Generate the embeddings (returns a NumPy array)
embeddings = model.encode(strings)


In [24]:
#Now if the record is already available in lanceDB if yes insert into duplicates or insert into original
# Search for the 5 most similar vectors
results = table.search(embeddings).limit(1).to_pandas()
if results.empty:
    strOriginalRecords.append(strRecord)
    table.add([
    {
        "hashkey": key,
        "vector": embeddings[0]
    }
])
else:
    originalkey=results.iloc[0]['hashkey']
    strDuplicateRecords.append([strRecord,originalkey])


In [25]:
print(strOriginalRecords)
print(strDuplicateRecords)

[['Sai', 'Charan', 'Anche', '22cd4f12ba00a3bc015bc835f159930451ed1541a5138287d89bb14fdf6e5002']]
[[['Sai', 'Charan', 'Anche', '22cd4f12ba00a3bc015bc835f159930451ed1541a5138287d89bb14fdf6e5002'], '22cd4f12ba00a3bc015bc835f159930451ed1541a5138287d89bb14fdf6e5002']]
